# Wildfire Burn Area Detection with the Segment Anything Model

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/segment-geospatial/blob/main/docs/examples/wildfire_burn_area.ipynb)

This notebook demonstrates how to use the [Segment Anything Model (SAM)](https://github.com/facebookresearch/segment-anything) to detect and delineate wildfire burn areas from satellite imagery. The workflow supports post-fire damage assessment and environmental monitoring pipelines.

**Study Area:** 2018 Camp Fire — Paradise, California, USA. One of the most destructive wildfires in California history, burning over 153,000 acres.

**Outline:**
- Install dependencies
- Download post-fire satellite imagery
- Run automatic mask generation with SamGeo
- Export burn area polygons to GeoJSON / Shapefile
- Visualize results on an interactive map

## Install dependencies

Uncomment the line below if running in Google Colab or a fresh environment.

In [ ]:
# %pip install segment-geospatial leafmap

## Import libraries

In [ ]:
import os
import leafmap
from samgeo import SamGeo, tms_to_geotiff

## Create an interactive map

Centre the map over the Camp Fire burn scar in Paradise, CA. You can pan and zoom to adjust the region of interest before downloading imagery.

In [ ]:
m = leafmap.Map(center=[39.758, -121.574], zoom=13)
m.add_basemap("SATELLITE")
m

## Define area of interest

The bounding box below covers the core burn perimeter of the 2018 Camp Fire near Paradise, CA. Adjust if you drew a custom rectangle on the map above.

In [ ]:
# [west, south, east, north] in WGS84 decimal degrees
bbox = [-121.64, 39.73, -121.50, 39.82]

# If you drew a rectangle on the map, replace bbox with:
# bbox = m.user_roi_bounds()

## Download satellite imagery

Convert map tiles to a GeoTIFF using the SATELLITE basemap. Zoom level 14 gives roughly 10 m/pixel resolution — sufficient to distinguish burn scars from surrounding vegetation.

In [ ]:
image_path = "campfire_postfire.tif"

tms_to_geotiff(
    output=image_path,
    bbox=bbox,
    zoom=14,
    source="Satellite",
    overwrite=True,
)

Verify the downloaded image on the map:

In [ ]:
m.add_raster(image_path, layer_name="Post-fire imagery")
m

## Initialize SamGeo

Load the ViT-H checkpoint. The model will be downloaded automatically on first run (~2.4 GB). A CUDA-enabled GPU is recommended but not required.

In [ ]:
sam = SamGeo(
    model_type="vit_h",
    automatic=True,
    sam_kwargs=None,
)

## Generate segmentation masks

Run automatic mask generation on the post-fire image. `batch=True` tiles the image into 512 × 512 px chips, which helps with large rasters and reduces GPU memory pressure.

In [ ]:
mask_path = "campfire_masks.tif"

sam.generate(
    source=image_path,
    output=mask_path,
    foreground=True,
    batch=True,
    batch_sample_size=(512, 512),
    erosion_kernel=(3, 3),
    mask_multiplier=255,
)

## Visualise masks

Display the generated masks overlaid on the source imagery.

In [ ]:
sam.show_anns(
    figsize=(14, 10),
    axis="off",
    alpha=0.4,
    output="campfire_annotated.png",
)

## Convert masks to vector polygons

Export the raster masks as GeoJSON and Shapefile for use in GIS workflows (QGIS, ArcGIS, GeoPandas, etc.).

In [ ]:
geojson_path = "campfire_burn_areas.geojson"
shapefile_path = "campfire_burn_areas.shp"

sam.tiff_to_geojson(
    tiff_path=mask_path,
    output=geojson_path,
    simplify_tolerance=None,
)

sam.tiff_to_shp(
    tiff_path=mask_path,
    output=shapefile_path,
    simplify_tolerance=None,
)

## Display results on an interactive map

Overlay the burn area polygons on the satellite basemap. Orange outlines highlight detected segments.

In [ ]:
style = {
    "color": "#FF6B35",
    "weight": 2,
    "fillColor": "#FF6B35",
    "fillOpacity": 0.3,
}

m.add_vector(
    geojson_path,
    layer_name="Burn Area Polygons",
    style=style,
)
m

## Summary

This notebook showed how to:

1. Download post-fire satellite imagery for a defined bounding box
2. Apply `SamGeo` automatic mask generation to segment burn areas
3. Export detected polygons as GeoJSON and Shapefile
4. Visualise results interactively with `leafmap`

The same pipeline can be applied to any wildfire event by changing `bbox` to match the target fire perimeter. For finer-grained burn severity analysis, consider combining this segmentation output with Sentinel-2 dNBR (differenced Normalized Burn Ratio) rasters.

## References

- [Segment Anything Model (SAM)](https://github.com/facebookresearch/segment-anything) — Kirillov et al., 2023
- [samgeo](https://samgeo.gishub.org/) — Wu et al., 2023
- [leafmap](https://leafmap.org/)
- [2018 Camp Fire — CAL FIRE](https://www.fire.ca.gov/incidents/2018/11/8/camp-fire/)